# 🧬 ZINC Database Batch Docking Pipeline - Google Colab
## ZINC Veritabanı ile Otomatik Batch Docking

**Özellikler:**
- ✅ ZINC veritabanından otomatik ligand indirme
- ✅ Google Drive entegrasyonu (kalıcılık)
- ✅ Devam etme özelliği (kesintiden kaldığı yerden devam)
- ✅ Otomatik grid box hesaplama
- ✅ Ligand validasyonu
- ✅ Güçlü hata yönetimi
- ✅ Kapsamlı log sistemi
- ✅ Bellek-optimize batch processing
- ✅ Kalite kontrol metrikleri
- ✅ SMILES/SDF'den PDBQT dönüştürme

**Yazar:** Ersin

**Tarih:** Aralık 2024

---

## 📌 Bölüm 1: Drive Bağlantısı ve Ortam Kurulumu

Bu bölüm her oturum başında çalıştırılmalıdır.

In [ ]:
# Google Drive'ı bağla
from google.colab import drive
import os
from pathlib import Path
import sys

print("🔌 Google Drive'a bağlanıyor...")
drive.mount('/content/drive')

# Kalıcı proje dizinini ayarla
PROJECT_ROOT = Path("/content/drive/MyDrive/ZINC_Docking_Project")
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

print(f"✅ Proje Dizini: {PROJECT_ROOT}")
print(f"📊 Kullanılabilir Alan: {os.statvfs(PROJECT_ROOT).f_bavail * os.statvfs(PROJECT_ROOT).f_frsize / (1024**3):.2f} GB")

In [ ]:
# Gerekli yazılımları kur (her runtime yeniden başlatıldığında çalıştır)
import subprocess

print("📦 Gerekli yazılımlar kuruluyor...\n")

# OpenBabel
if not os.path.exists("/usr/bin/obabel"):
    print("OpenBabel kuruluyor...")
    !apt-get update -qq
    !apt-get install -qq -y openbabel
    print("  ✓ OpenBabel kuruldu")
else:
    print("  ✓ OpenBabel zaten kurulu")

# AutoDock Vina
if not os.path.exists("/usr/local/bin/vina"):
    print("AutoDock Vina kuruluyor...")
    !wget -q https://github.com/ccsb-scripps/AutoDock-Vina/releases/download/v1.2.5/vina_1.2.5_linux_x86_64 -O /tmp/vina
    !chmod +x /tmp/vina
    !sudo mv /tmp/vina /usr/local/bin/vina
    print("  ✓ Vina 1.2.5 kuruldu")
else:
    print("  ✓ Vina zaten kurulu")

# Vina versiyonunu kontrol et
vina_check = subprocess.run(['vina', '--version'], capture_output=True, text=True)
print(f"\n📍 Vina Versiyonu: {vina_check.stdout.strip()}")

# Python paketleri
print("\nPython paketleri kuruluyor...")
!pip install -q rdkit biopython pandas numpy tqdm openpyxl matplotlib seaborn requests

print("\n🎉 Tüm yazılımlar başarıyla kuruldu!")

## 📁 Bölüm 2: Dizin Yapısı ve Dosya Yönetimi

In [ ]:
import shutil
import logging
from datetime import datetime

# Dizin yapısını oluştur
ligands_dir = PROJECT_ROOT / "ligands"
zinc_download_dir = ligands_dir / "zinc_raw"  # ZINC'den indirilenler
sdf_dir = ligands_dir / "sdf_files"  # SDF formatı
pdbqt_dir = ligands_dir / "pdbqt_final"  # PDBQT formatı
target_dir = PROJECT_ROOT / "target"
results_dir = PROJECT_ROOT / "results"
logs_dir = PROJECT_ROOT / "logs"

for d in [ligands_dir, zinc_download_dir, sdf_dir, pdbqt_dir, target_dir, results_dir, logs_dir]:
    d.mkdir(parents=True, exist_ok=True)

# Log sistemini kur
log_file = logs_dir / f"docking_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(log_file),
        logging.StreamHandler(sys.stdout)
    ]
)

logger = logging.getLogger(__name__)
logger.info(f"📂 Dizin yapısı oluşturuldu: {PROJECT_ROOT}")
logger.info(f"📝 Log dosyası: {log_file}")

print(f"\n✅ Dizinler hazır!")

## 🔬 Bölüm 3: ZINC Veritabanı Entegrasyonu

Bu bölüm ZINC veritabanından ligand indirme fonksiyonlarını içerir.

In [ ]:
import requests
import time
from typing import List, Optional
import json

class ZINCDownloader:
    """
    ZINC veritabanından ligand indirme sınıfı.
    
    ZINC15 API kullanır.
    """
    
    def __init__(self, output_dir: Path):
        self.output_dir = output_dir
        self.base_url = "https://zinc15.docking.org"
        self.session = requests.Session()
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Molecular Docking Research)'
        })
    
    def search_by_smiles(self, smiles: str, similarity: float = 0.8) -> List[str]:
        """
        SMILES ile benzerlik araması yap.
        
        Parameters:
        -----------
        smiles : str
            Aranacak SMILES yapısı
        similarity : float
            Benzerlik eşiği (0-1)
        
        Returns:
        --------
        List[str] : ZINC ID listesi
        """
        url = f"{self.base_url}/substances/search/"
        params = {
            'q': smiles,
            'similarity': similarity
        }
        
        try:
            response = self.session.get(url, params=params, timeout=30)
            response.raise_for_status()
            
            # Parse response (bu ZINC API'ye göre değişebilir)
            data = response.json()
            zinc_ids = [item['zinc_id'] for item in data.get('results', [])]
            
            logger.info(f"SMILES araması: {len(zinc_ids)} bileşik bulundu")
            return zinc_ids
            
        except Exception as e:
            logger.error(f"ZINC arama hatası: {str(e)}")
            return []
    
    def download_zinc_ids(self, zinc_ids: List[str], format: str = 'sdf') -> List[Path]:
        """
        ZINC ID'lerinden ligandları indir.
        
        Parameters:
        -----------
        zinc_ids : List[str]
            İndirilecek ZINC ID listesi
        format : str
            Dosya formatı ('sdf', 'mol2', 'smiles')
        
        Returns:
        --------
        List[Path] : İndirilen dosya yolları
        """
        downloaded = []
        
        for zinc_id in tqdm(zinc_ids, desc="ZINC indiriliyor"):
            try:
                # 3D yapıyı indir (docking için gerekli)
                url = f"{self.base_url}/substances/{zinc_id}.{format}"
                
                response = self.session.get(url, timeout=30)
                response.raise_for_status()
                
                # Dosyayı kaydet
                output_file = self.output_dir / f"{zinc_id}.{format}"
                with open(output_file, 'wb') as f:
                    f.write(response.content)
                
                downloaded.append(output_file)
                
                # Rate limiting (ZINC sunucusunu aşırı yüklememek için)
                time.sleep(0.5)
                
            except Exception as e:
                logger.warning(f"{zinc_id} indirilemedi: {str(e)}")
                continue
        
        logger.info(f"✓ {len(downloaded)}/{len(zinc_ids)} ligand indirildi")
        return downloaded
    
    def download_subset(self, subset_name: str, max_molecules: int = 1000) -> List[Path]:
        """
        ZINC subset'inden molekül indir.
        
        Popüler subsetler:
        - 'fda': FDA onaylı ilaçlar
        - 'world': Dünya çapında onaylı ilaçlar  
        - 'investigational': Araştırma aşamasındaki ilaçlar
        - 'lead-like': Lead-like moleküller
        - 'fragment': Fragment library
        
        Parameters:
        -----------
        subset_name : str
            Subset adı
        max_molecules : int
            Maksimum indirilecek molekül sayısı
        
        Returns:
        --------
        List[Path] : İndirilen dosyalar
        """
        logger.info(f"ZINC subset indiriliyor: {subset_name} (max: {max_molecules})")
        
        # ZINC subset URL'si (gerçek API endpoint'e göre ayarlanmalı)
        url = f"{self.base_url}/catalogs/subsets/{subset_name}.txt"
        
        try:
            response = self.session.get(url, timeout=60)
            response.raise_for_status()
            
            # ZINC ID'lerini parse et
            zinc_ids = response.text.strip().split('\n')[:max_molecules]
            
            logger.info(f"Subset'den {len(zinc_ids)} ID alındı")
            
            # İndir
            return self.download_zinc_ids(zinc_ids, format='sdf')
            
        except Exception as e:
            logger.error(f"Subset indirme hatası: {str(e)}")
            return []
    
    def download_from_file(self, zinc_id_file: Path) -> List[Path]:
        """
        Dosyadan ZINC ID'leri oku ve indir.
        
        Dosya formatı: Her satırda bir ZINC ID
        Örnek:
        ZINC000000001
        ZINC000000002
        ...
        
        Parameters:
        -----------
        zinc_id_file : Path
            ZINC ID'leri içeren dosya
        
        Returns:
        --------
        List[Path] : İndirilen dosyalar
        """
        logger.info(f"ZINC ID dosyası okunuyor: {zinc_id_file}")
        
        with open(zinc_id_file, 'r') as f:
            zinc_ids = [line.strip() for line in f if line.strip() and not line.startswith('#')]
        
        logger.info(f"{len(zinc_ids)} ZINC ID bulundu")
        
        return self.download_zinc_ids(zinc_ids, format='sdf')

print("✅ ZINC indirici sınıfı yüklendi")

## 🎯 Bölüm 3B: Alternatif - Basitleştirilmiş ZINC İndirme

Eğer ZINC API'ye erişim sorunları yaşıyorsanız, bu alternatif yöntemi kullanın.

In [ ]:
def download_zinc_simple(zinc_ids: List[str], output_dir: Path) -> List[Path]:
    """
    Basitleştirilmiş ZINC indirme fonksiyonu.
    Doğrudan ZINC15'ten 3D SDF formatında indirir.
    """
    downloaded = []
    
    print(f"🔽 {len(zinc_ids)} ZINC molekülü indiriliyor...")
    
    for zinc_id in tqdm(zinc_ids, desc="İndiriliyor"):
        try:
            # ZINC15 direkt download URL'si
            # Format: https://zinc15.docking.org/substances/ZINCID.sdf
            url = f"https://zinc15.docking.org/substances/{zinc_id}.sdf"
            
            response = requests.get(url, timeout=30)
            
            if response.status_code == 200:
                output_file = output_dir / f"{zinc_id}.sdf"
                with open(output_file, 'wb') as f:
                    f.write(response.content)
                downloaded.append(output_file)
            else:
                logger.warning(f"{zinc_id}: HTTP {response.status_code}")
            
            # Rate limiting
            time.sleep(1)  # 1 saniye bekle
            
        except Exception as e:
            logger.error(f"{zinc_id} hatası: {str(e)}")
            continue
    
    print(f"✅ {len(downloaded)} molekül başarıyla indirildi")
    return downloaded

# Örnek kullanım:
# zinc_ids = ['ZINC000000001', 'ZINC000000002', 'ZINC000000003']
# files = download_zinc_simple(zinc_ids, zinc_download_dir)

print("✅ Basit ZINC indirme fonksiyonu hazır")

## 🧪 Bölüm 4: Ligand Hazırlama (SDF → PDBQT)

ZINC'den indirilen SDF dosyalarını docking için PDBQT formatına çevirir.

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors

def prepare_ligand_from_sdf(sdf_file: Path, output_dir: Path) -> Optional[Path]:
    """
    SDF dosyasından PDBQT dosyası oluştur.
    
    İşlemler:
    1. 3D koordinat kontrolü ve gerekirse üretimi
    2. Hidrojen ekleme (pH 7.4)
    3. Gasteiger yükleri ekleme
    4. PDBQT formatına dönüştürme
    
    Parameters:
    -----------
    sdf_file : Path
        Giriş SDF dosyası
    output_dir : Path
        Çıkış dizini
    
    Returns:
    --------
    Optional[Path] : PDBQT dosya yolu veya None
    """
    try:
        # Molekülü oku
        suppl = Chem.SDMolSupplier(str(sdf_file))
        mol = next(suppl)
        
        if mol is None:
            logger.warning(f"Molekül okunamadı: {sdf_file.name}")
            return None
        
        # 3D koordinat kontrolü
        if mol.GetNumConformers() == 0:
            logger.info(f"3D koordinat üretiliyor: {sdf_file.name}")
            # 3D koordinat üret
            mol = Chem.AddHs(mol)
            AllChem.EmbedMolecule(mol, randomSeed=42)
            AllChem.MMFFOptimizeMolecule(mol)
        
        # Geçici PDB dosyası oluştur
        temp_pdb = output_dir / f"{sdf_file.stem}_temp.pdb"
        Chem.MolToPDBFile(mol, str(temp_pdb))
        
        # PDBQT'ye çevir (OpenBabel ile)
        pdbqt_file = output_dir / f"{sdf_file.stem}.pdbqt"
        
        cmd = [
            'obabel',
            str(temp_pdb),
            '-O', str(pdbqt_file),
            '-xr',  # Önce tüm hidrojenleri kaldır
            '-p', '7.4',  # pH 7.4'te hidrojen ekle
            '--partialcharge', 'gasteiger'  # Gasteiger yükleri ekle
        ]
        
        result = subprocess.run(cmd, capture_output=True, text=True)
        
        # Geçici dosyayı sil
        temp_pdb.unlink()
        
        if result.returncode == 0 and pdbqt_file.exists():
            return pdbqt_file
        else:
            logger.error(f"PDBQT dönüşümü başarısız: {sdf_file.name}")
            return None
            
    except Exception as e:
        logger.error(f"Ligand hazırlama hatası ({sdf_file.name}): {str(e)}")
        return None

def batch_prepare_ligands(input_dir: Path, output_dir: Path) -> List[Path]:
    """
    Tüm SDF dosyalarını toplu olarak PDBQT'ye çevir.
    """
    sdf_files = list(input_dir.glob("*.sdf"))
    logger.info(f"🔄 {len(sdf_files)} SDF dosyası PDBQT'ye dönüştürülüyor...")
    
    pdbqt_files = []
    
    for sdf_file in tqdm(sdf_files, desc="Dönüştürülüyor"):
        pdbqt_file = prepare_ligand_from_sdf(sdf_file, output_dir)
        if pdbqt_file:
            pdbqt_files.append(pdbqt_file)
    
    logger.info(f"✅ {len(pdbqt_files)}/{len(sdf_files)} ligand başarıyla hazırlandı")
    
    return pdbqt_files

print("✅ Ligand hazırlama fonksiyonları yüklendi")

## 🎯 Bölüm 5: Yardımcı Fonksiyonlar

In [ ]:
import multiprocessing
from Bio.PDB import PDBParser, PDBIO, Select
import numpy as np
from tqdm import tqdm

# CPU tespiti
def get_optimal_cpu_count():
    """Docking için optimal CPU sayısını belirle"""
    total_cpus = multiprocessing.cpu_count()
    # Sistem için 1 CPU ayır
    return max(1, total_cpus - 1)

# Ligand validasyonu
def validate_pdbqt(pdbqt_path):
    """
    PDBQT dosya formatını ve içeriğini kontrol et.
    
    Kontroller:
    - Dosya varlığı ve okunabilirliği
    - ROOT/ENDROOT blokları
    - ATOM/HETATM kayıtları
    - 3D koordinatlar
    - Minimum boyut kontrolü
    """
    try:
        if not pdbqt_path.exists():
            return False, "Dosya bulunamadı"
        
        if pdbqt_path.stat().st_size < 100:  # Çok küçük
            return False, "Dosya çok küçük"
        
        with open(pdbqt_path, 'r') as f:
            content = f.read()
            
            # Temel anahtar kelimeleri kontrol et
            if 'ROOT' not in content or 'ENDROOT' not in content:
                return False, "ROOT/ENDROOT blokları eksik"
            
            if 'ATOM' not in content and 'HETATM' not in content:
                return False, "ATOM/HETATM kayıtları yok"
            
            # Koordinat kontrolü
            lines = content.split('\n')
            atom_lines = [l for l in lines if l.startswith('ATOM') or l.startswith('HETATM')]
            
            if len(atom_lines) < 3:  # En az 3 atom
                return False, "Çok az atom"
            
            return True, "Geçerli"
            
    except Exception as e:
        return False, f"Hata: {str(e)}"

# Grid box hesaplama
def calculate_grid_box(pdb_path, residue_list=None, padding=10.0):
    """
    Bağlanma bölgesi veya tüm proteinden grid box hesapla.
    
    Parametreler:
    -----------
    pdb_path : Path
        PDB dosya yolu
    residue_list : list, optional
        Bağlanma bölgesi residue listesi
        Örnek: [('A', 123), ('A', 456)]
    padding : float
        Bağlanma bölgesi etrafındaki ekstra alan (Angstrom)
    
    Dönüş:
    --------
    dict : Vina için grid parametreleri
    """
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure('protein', str(pdb_path))
    
    coords = []
    
    if residue_list:
        # Belirtilen residue'leri kullan
        for chain_id, res_id in residue_list:
            try:
                chain = structure[0][chain_id]
                residue = chain[res_id]
                for atom in residue:
                    coords.append(atom.get_coord())
            except KeyError:
                logger.warning(f"Residue {chain_id}:{res_id} bulunamadı")
    else:
        # Tüm proteini kullan
        for atom in structure.get_atoms():
            coords.append(atom.get_coord())
    
    if not coords:
        raise ValueError("Grid hesaplaması için koordinat bulunamadı")
    
    coords = np.array(coords)
    
    # Merkezi hesapla
    center = coords.mean(axis=0)
    
    # Padding ile boyutu hesapla
    size = coords.max(axis=0) - coords.min(axis=0) + 2 * padding
    
    # Maksimum boyutu sınırla (Vina kısıtlaması)
    size = np.minimum(size, 50.0)
    
    grid_params = {
        'center_x': float(center[0]),
        'center_y': float(center[1]),
        'center_z': float(center[2]),
        'size_x': float(size[0]),
        'size_y': float(size[1]),
        'size_z': float(size[2])
    }
    
    logger.info(f"Grid Merkezi: ({center[0]:.3f}, {center[1]:.3f}, {center[2]:.3f})")
    logger.info(f"Grid Boyutu: ({size[0]:.1f} x {size[1]:.1f} x {size[2]:.1f}) Å")
    
    return grid_params

print("✅ Yardımcı fonksiyonlar yüklendi")
print(f"💻 Kullanılabilir CPU: {multiprocessing.cpu_count()}")
print(f"🎯 Docking için {get_optimal_cpu_count()} CPU kullanılacak")

## 🧬 Bölüm 6: Hedef Protein Hazırlama

In [ ]:
from Bio.PDB import PDBList

def prepare_target_protein(pdb_code, custom_grid_residues=None):
    """
    Hedef proteini indir, temizle ve hazırla.
    
    Parametreler:
    -----------
    pdb_code : str
        PDB ID (örn: '7EU8')
    custom_grid_residues : list, optional
        Bağlanma bölgesi için (chain, resid) listesi
    
    Dönüş:
    --------
    tuple : (pdbqt_path, grid_params)
    """
    target_pdbqt = target_dir / f"{pdb_code}.pdbqt"
    
    # Zaten hazırlanmış mı kontrol et
    if target_pdbqt.exists():
        logger.info(f"✅ Hedef protein zaten hazır: {target_pdbqt}")
        
        # Grid parametrelerini hesapla veya yükle
        clean_pdb = target_dir / f"{pdb_code}_clean.pdb"
        if clean_pdb.exists():
            grid_params = calculate_grid_box(clean_pdb, custom_grid_residues)
            return target_pdbqt, grid_params
        else:
            logger.warning("Temiz PDB bulunamadı, varsayılan grid kullanılacak")
            return target_pdbqt, None
    
    logger.info(f"⬇️ Hedef protein indiriliyor: {pdb_code}")
    
    # İndir
    pdbl = PDBList()
    pdb_file = pdbl.retrieve_pdb_file(pdb_code, pdir=str(target_dir), file_format='pdb')
    
    # Temizle (su ve protein olmayan atomları kaldır)
    logger.info("🧹 Yapı temizleniyor (su ve protein olmayan atomlar kaldırılıyor)")
    clean_pdb = target_dir / f"{pdb_code}_clean.pdb"
    
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure('target', pdb_file)
    
    class ProteinSelect(Select):
        def accept_residue(self, residue):
            # Sadece standart amino asitleri tut
            return residue.id[0] == ' '
    
    io = PDBIO()
    io.set_structure(structure)
    io.save(str(clean_pdb), ProteinSelect())
    
    # PDBQT'ye çevir
    logger.info("🔄 Gasteiger yükleri ile PDBQT formatına dönüştürülüyor")
    cmd = [
        'obabel', str(clean_pdb), '-O', str(target_pdbqt),
        '-xr',  # Önce tüm hidrojenleri kaldır
        '-p', '7.4',  # pH 7.4'te hidrojen ekle
        '--partialcharge', 'gasteiger'  # Gasteiger yükleri ekle
    ]
    
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    if result.returncode != 0:
        logger.error(f"OpenBabel dönüşümü başarısız: {result.stderr}")
        raise RuntimeError("Hedef hazırlama başarısız")
    
    logger.info(f"✓ Hedef hazırlandı ve kaydedildi: {target_pdbqt}")
    
    # Grid hesapla
    grid_params = calculate_grid_box(clean_pdb, custom_grid_residues)
    
    return target_pdbqt, grid_params

# Örnek kullanım - ihtiyacınıza göre değiştirin
TARGET_PDB = "7EU8"  # Hedef protein PDB ID'nizi buraya girin

# Seçenek 1: Tüm proteini grid için kullan
target_path, grid_params = prepare_target_protein(TARGET_PDB)

# Seçenek 2: Belirli bağlanma bölgesi residue'lerini kullan (yorumu kaldırıp değiştirin)
# binding_site = [('A', 123), ('A', 456), ('A', 789)]  # Örnek residue'ler
# target_path, grid_params = prepare_target_protein(TARGET_PDB, binding_site)

print(f"\n✅ Hedef hazır: {target_path.name}")
print(f"📐 Grid parametreleri hesaplandı")

## 🚀 Bölüm 7: Docking Motor Fonksiyonları

In [ ]:
import pandas as pd

def run_vina_docking(receptor, ligand, output, params, timeout=180):
    """
    AutoDock Vina docking işlemini çalıştır.
    
    Parametreler:
    -----------
    receptor : Path
        Reseptör PDBQT dosyası
    ligand : Path
        Ligand PDBQT dosyası
    output : Path
        Çıktı PDBQT dosyası (docked pozlar)
    params : dict
        Grid parametreleri ve ayarlar
    timeout : int
        Docking başına maksimum süre (saniye)
    
    Dönüş:
    --------
    tuple : (best_affinity, num_poses, rmsd_ub) veya (None, 0, 0)
    """
    cmd = [
        'vina',
        '--receptor', str(receptor),
        '--ligand', str(ligand),
        '--out', str(output),
        '--center_x', str(params['center_x']),
        '--center_y', str(params['center_y']),
        '--center_z', str(params['center_z']),
        '--size_x', str(params['size_x']),
        '--size_y', str(params['size_y']),
        '--size_z', str(params['size_z']),
        '--exhaustiveness', str(params['exhaustiveness']),
        '--cpu', str(params['cpu']),
        '--num_modes', '9'  # 9'a kadar poz üret
    ]
    
    try:
        proc = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            timeout=timeout
        )
        
        if proc.returncode == 0:
            # Affinity ve RMSD için çıktıyı parse et
            lines = proc.stdout.split('\n')
            
            for line in lines:
                parts = line.split()
                if len(parts) > 3 and parts[0] == '1':  # En iyi poz
                    affinity = float(parts[1])
                    rmsd_lb = float(parts[2])
                    rmsd_ub = float(parts[3])
                    
                    # Poz sayısını say
                    num_poses = sum(1 for l in lines if l.strip() and l.split()[0].isdigit())
                    
                    return affinity, num_poses, rmsd_ub
            
            logger.warning(f"Vina çıktısı parse edilemedi: {ligand.name}")
            return None, 0, 0
        else:
            logger.error(f"Vina başarısız ({ligand.name}): {proc.stderr[:200]}")
            return None, 0, 0
            
    except subprocess.TimeoutExpired:
        logger.warning(f"Timeout ({timeout}s): {ligand.name}")
        return None, 0, 0
    except Exception as e:
        logger.error(f"Beklenmeyen hata ({ligand.name}): {str(e)}")
        return None, 0, 0

print("✅ Docking motoru yüklendi")

## 🎬 Bölüm 8: ZINC İndirme ve Ligand Hazırlama İŞ AKIŞI

Bu bölümde ZINC'den ligand indirip PDBQT formatına çevireceğiz.

In [ ]:
# ==================== ZINC İNDİRME AYARLARI ====================

# Seçenek 1: Manuel ZINC ID listesi
MANUAL_ZINC_IDS = [
    # Buraya ZINC ID'lerinizi ekleyin
    # Örnek:
    # 'ZINC000000001',
    # 'ZINC000000002',
    # 'ZINC000000003',
]

# Seçenek 2: Dosyadan ZINC ID'leri oku
ZINC_ID_FILE = None  # Örnek: PROJECT_ROOT / "zinc_ids.txt"

# Seçenek 3: ZINC subset kullan (FDA onaylı ilaçlar, vb.)
USE_ZINC_SUBSET = True
ZINC_SUBSET_NAME = "fda"  # 'fda', 'world', 'investigational', 'lead-like', 'fragment'
MAX_MOLECULES_FROM_SUBSET = 100  # Subset'ten maksimum kaç molekül indirilecek

print("⚙️ ZINC indirme ayarları:")
if USE_ZINC_SUBSET:
    print(f"  📦 Subset: {ZINC_SUBSET_NAME}")
    print(f"  🔢 Maksimum molekül: {MAX_MOLECULES_FROM_SUBSET}")
elif ZINC_ID_FILE:
    print(f"  📄 Dosyadan: {ZINC_ID_FILE}")
elif MANUAL_ZINC_IDS:
    print(f"  ✍️ Manuel liste: {len(MANUAL_ZINC_IDS)} ID")
else:
    print("  ⚠️ UYARI: Hiçbir ZINC kaynağı seçilmedi!")

In [ ]:
# ==================== ZINC İNDİRME İŞLEMİ ====================

logger.info("🔽 ZINC molekül indirme başlıyor...")

# ZINC downloader oluştur
zinc_downloader = ZINCDownloader(zinc_download_dir)

downloaded_files = []

# Hangi yöntem seçildiyse ona göre indir
if USE_ZINC_SUBSET:
    logger.info(f"📦 ZINC subset indiriliyor: {ZINC_SUBSET_NAME}")
    downloaded_files = zinc_downloader.download_subset(
        ZINC_SUBSET_NAME, 
        max_molecules=MAX_MOLECULES_FROM_SUBSET
    )
    
elif ZINC_ID_FILE and ZINC_ID_FILE.exists():
    logger.info(f"📄 ZINC ID dosyasından indiriliyor: {ZINC_ID_FILE}")
    downloaded_files = zinc_downloader.download_from_file(ZINC_ID_FILE)
    
elif MANUAL_ZINC_IDS:
    logger.info(f"✍️ Manuel ZINC ID listesinden indiriliyor: {len(MANUAL_ZINC_IDS)} molekül")
    downloaded_files = zinc_downloader.download_zinc_ids(MANUAL_ZINC_IDS, format='sdf')
    
else:
    logger.error("❌ Hiçbir ZINC kaynağı seçilmedi! Lütfen ayarları kontrol edin.")
    raise ValueError("ZINC kaynağı belirtilmedi")

logger.info(f"✅ {len(downloaded_files)} ZINC molekülü indirildi")
print(f"\n📊 İndirilen dosyalar: {zinc_download_dir}")

In [ ]:
# ==================== LIGAND HAZIRLAMA (SDF → PDBQT) ====================

logger.info("🔄 İndirilen ligandlar PDBQT formatına dönüştürülüyor...")

# Batch dönüştürme
prepared_ligands = batch_prepare_ligands(zinc_download_dir, pdbqt_dir)

logger.info(f"✅ {len(prepared_ligands)} ligand PDBQT formatına dönüştürüldü")
print(f"\n📊 PDBQT dosyaları: {pdbqt_dir}")
print(f"✅ Docking için {len(prepared_ligands)} ligand hazır!")

## 🎯 Bölüm 9: Ana Docking Pipeline (Devam Etme Özelliği ile)

ZINC'den indirilen ligandlarla batch docking.

In [ ]:
# ==================== DOCKING AYARLARI ====================

# Çıktı dosyaları
CSV_LOG_PATH = PROJECT_ROOT / "DOCKING_RESULTS.csv"
FAILED_LOG_PATH = PROJECT_ROOT / "FAILED_LIGANDS.csv"

# Performans ayarları
SAVE_INTERVAL = 10  # Her N ligandda bir sonuçları kaydet
BATCH_SIZE = 100    # Her N ligandda bir belleği temizle

# Docking parametreleri
DOCKING_PARAMS = {
    'exhaustiveness': 16,  # Yüksek kalite (8=hızlı, 16=dengeli, 32=detaylı)
    'cpu': get_optimal_cpu_count(),
    'timeout': 180  # Ligand başına 3 dakika
}

# Grid parametreleriyle birleştir
if grid_params:
    DOCKING_PARAMS.update(grid_params)
else:
    logger.error("Grid parametreleri mevcut değil! Lütfen önce Bölüm 6'yı çalıştırın.")
    raise ValueError("Grid parametreleri eksik")

# Kalite eşikleri
GOOD_AFFINITY_THRESHOLD = -7.0  # kcal/mol
EXCELLENT_AFFINITY_THRESHOLD = -9.0  # kcal/mol

print("⚙️ Docking Ayarları:")
print(f"  Exhaustiveness: {DOCKING_PARAMS['exhaustiveness']}")
print(f"  CPU çekirdekleri: {DOCKING_PARAMS['cpu']}")
print(f"  Timeout: {DOCKING_PARAMS['timeout']}s per ligand")
print(f"  Kaydetme aralığı: {SAVE_INTERVAL} ligand")
print(f"  Batch boyutu: {BATCH_SIZE} ligand")

In [ ]:
# ==================== ANA DOCKING PİPELİNE ====================

logger.info("🚀 Docking pipeline başlatılıyor...")
logger.info(f"Sonuçlar şuraya kaydedilecek: {CSV_LOG_PATH}")

# Adım 1: Ligandları topla ve valide et
logger.info("📋 Ligand dizini taranıyor...")
ligand_files = list(pdbqt_dir.glob("*.pdbqt"))
total_ligands = len(ligand_files)
logger.info(f"{total_ligands} PDBQT dosyası bulundu")

# Ligandları valide et
logger.info("✅ Ligand dosyaları valide ediliyor...")
valid_ligands = []
invalid_ligands = []

for lig_file in tqdm(ligand_files, desc="Valide ediliyor"):
    is_valid, msg = validate_pdbqt(lig_file)
    if is_valid:
        valid_ligands.append(lig_file)
    else:
        invalid_ligands.append((lig_file.name, msg))
        logger.warning(f"Geçersiz ligand {lig_file.name}: {msg}")

logger.info(f"✓ Geçerli ligandlar: {len(valid_ligands)}")
logger.info(f"✗ Geçersiz ligandlar: {len(invalid_ligands)}")

if invalid_ligands:
    # Geçersiz ligandlar raporunu kaydet
    pd.DataFrame(invalid_ligands, columns=['Ligand', 'Sebep']).to_csv(
        PROJECT_ROOT / "INVALID_LIGANDS.csv", index=False
    )

# Adım 2: Önceki sonuçları yükle (Devam etme özelliği)
results_data = []
failed_data = []
processed_names = set()

if CSV_LOG_PATH.exists():
    df_existing = pd.read_csv(CSV_LOG_PATH)
    results_data = df_existing.to_dict('records')
    processed_names = set(df_existing['Ligand_Name'].tolist())
    logger.info(f"♻️ {len(processed_names)} önceki sonuç yüklendi")
    logger.info("Son kontrol noktasından devam ediliyor...")
else:
    logger.info("🆕 Yeni tarama başlatılıyor")

if FAILED_LOG_PATH.exists():
    df_failed = pd.read_csv(FAILED_LOG_PATH)
    failed_data = df_failed.to_dict('records')

# Adım 3: Docking döngüsü
logger.info("\n🧬 Moleküler docking başlıyor...")
start_time = time.time()
counter = 0
success_count = 0
fail_count = 0

pbar = tqdm(
    valid_ligands,
    desc="Docking İlerlemesi",
    unit="ligand",
    initial=len(processed_names)
)

for lig_file in pbar:
    name = lig_file.stem
    
    # Zaten işlenmişse atla
    if name in processed_names:
        continue
    
    out_file = results_dir / f"{name}_docked.pdbqt"
    
    # Docking çalıştır
    affinity, num_poses, rmsd = run_vina_docking(
        target_path,
        lig_file,
        out_file,
        DOCKING_PARAMS,
        timeout=DOCKING_PARAMS['timeout']
    )
    
    # Sonuçları kaydet
    if affinity is not None:
        # Sonuç kalitesini sınıflandır
        if affinity <= EXCELLENT_AFFINITY_THRESHOLD:
            quality = "Mükemmel"
        elif affinity <= GOOD_AFFINITY_THRESHOLD:
            quality = "İyi"
        else:
            quality = "Zayıf"
        
        results_data.append({
            'Ligand_Name': name,
            'Affinity_kcal_mol': affinity,
            'Num_Poses': num_poses,
            'RMSD_ub': rmsd,
            'Status': 'Başarılı',
            'Quality': quality,
            'Output_File': out_file.name
        })
        
        success_count += 1
        pbar.set_postfix({
            'Skor': f"{affinity:.2f}",
            'Kalite': quality,
            'Başarılı': success_count,
            'Başarısız': fail_count
        })
        
    else:
        # Başarısızlığı kaydet
        failed_data.append({
            'Ligand_Name': name,
            'Status': 'Başarısız',
            'Timestamp': datetime.now().isoformat()
        })
        
        fail_count += 1
        pbar.set_postfix({
            'Başarılı': success_count,
            'Başarısız': fail_count
        })
    
    counter += 1
    
    # Periyodik kayıt (her SAVE_INTERVAL ligandda)
    if counter % SAVE_INTERVAL == 0:
        pd.DataFrame(results_data).to_csv(CSV_LOG_PATH, index=False)
        if failed_data:
            pd.DataFrame(failed_data).to_csv(FAILED_LOG_PATH, index=False)
        logger.info(f"💾 İlerleme kaydedildi: {counter} ligand işlendi")
    
    # Bellek optimizasyonu (her BATCH_SIZE ligandda)
    if counter % BATCH_SIZE == 0:
        # Kaydet ve belleği boşaltmak için yeniden yükle
        df_temp = pd.DataFrame(results_data)
        df_temp.to_csv(CSV_LOG_PATH, index=False)
        del df_temp
        logger.info(f"🧹 Bellek temizlendi ({counter} ligand)")

# Son kayıt
logger.info("\n💾 Final sonuçlar kaydediliyor...")
df_final = pd.DataFrame(results_data)
df_final.to_csv(CSV_LOG_PATH, index=False)

if failed_data:
    df_failed_final = pd.DataFrame(failed_data)
    df_failed_final.to_csv(FAILED_LOG_PATH, index=False)

# Özet istatistikler
elapsed_time = time.time() - start_time
logger.info("\n" + "="*60)
logger.info("🎉 DOCKING TAMAMLANDI!")
logger.info("="*60)
logger.info(f"İşlenen toplam ligand: {counter}")
logger.info(f"Başarılı docking: {success_count}")
logger.info(f"Başarısız docking: {fail_count}")
logger.info(f"Başarı oranı: {100*success_count/(success_count+fail_count):.1f}%")
logger.info(f"Toplam süre: {elapsed_time/3600:.2f} saat")
logger.info(f"Ligand başına ortalama süre: {elapsed_time/counter:.1f} saniye")
logger.info(f"\n📊 Sonuçlar: {CSV_LOG_PATH}")
logger.info(f"📝 Log dosyası: {log_file}")

if success_count > 0:
    df_stats = df_final[df_final['Status'] == 'Başarılı']
    logger.info("\n📈 Affinity İstatistikleri:")
    logger.info(f"  En iyi affinity: {df_stats['Affinity_kcal_mol'].min():.2f} kcal/mol")
    logger.info(f"  Ortalama affinity: {df_stats['Affinity_kcal_mol'].mean():.2f} kcal/mol")
    logger.info(f"  Medyan affinity: {df_stats['Affinity_kcal_mol'].median():.2f} kcal/mol")
    logger.info(f"  Mükemmel bağlanma (<{EXCELLENT_AFFINITY_THRESHOLD}): {(df_stats['Affinity_kcal_mol'] <= EXCELLENT_AFFINITY_THRESHOLD).sum()}")
    logger.info(f"  İyi bağlanma (<{GOOD_AFFINITY_THRESHOLD}): {(df_stats['Affinity_kcal_mol'] <= GOOD_AFFINITY_THRESHOLD).sum()}")

print("\n✅ Tüm sonuçlar Google Drive'a güvenle kaydedildi!")

## 📊 Bölüm 10: Sonuç Analizi ve Görselleştirme

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Sonuçları yükle
if CSV_LOG_PATH.exists():
    df_results = pd.read_csv(CSV_LOG_PATH)
    df_success = df_results[df_results['Status'] == 'Başarılı'].copy()
    
    print(f"📊 {len(df_success)} başarılı docking sonucu analiz ediliyor...\n")
    
    # Stil ayarla
    sns.set_style("whitegrid")
    plt.rcParams['figure.figsize'] = (14, 10)
    
    # Alt grafiklerde figure oluştur
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # 1. Affinity dağılımı
    axes[0, 0].hist(df_success['Affinity_kcal_mol'], bins=50, edgecolor='black', alpha=0.7)
    axes[0, 0].axvline(GOOD_AFFINITY_THRESHOLD, color='orange', linestyle='--', 
                       label=f'İyi (<{GOOD_AFFINITY_THRESHOLD})')
    axes[0, 0].axvline(EXCELLENT_AFFINITY_THRESHOLD, color='red', linestyle='--', 
                       label=f'Mükemmel (<{EXCELLENT_AFFINITY_THRESHOLD})')
    axes[0, 0].set_xlabel('Bağlanma Afinitesi (kcal/mol)')
    axes[0, 0].set_ylabel('Frekans')
    axes[0, 0].set_title('Bağlanma Affinitelerinin Dağılımı')
    axes[0, 0].legend()
    
    # 2. Top 20 ligandlar
    top20 = df_success.nsmallest(20, 'Affinity_kcal_mol')
    axes[0, 1].barh(range(len(top20)), top20['Affinity_kcal_mol'].values)
    axes[0, 1].set_yticks(range(len(top20)))
    axes[0, 1].set_yticklabels(top20['Ligand_Name'].values, fontsize=8)
    axes[0, 1].set_xlabel('Affinity (kcal/mol)')
    axes[0, 1].set_title('En İyi 20 Ligand (Bağlanma Affinitesine Göre)')
    axes[0, 1].invert_yaxis()
    
    # 3. Kalite dağılımı
    quality_counts = df_success['Quality'].value_counts()
    axes[1, 0].pie(quality_counts.values, labels=quality_counts.index, autopct='%1.1f%%',
                   colors=['#ff6b6b', '#ffd93d', '#6bcf7f'])
    axes[1, 0].set_title('Kalite Dağılımı')
    
    # 4. Affinity vs Poz Sayısı
    scatter = axes[1, 1].scatter(df_success['Num_Poses'], 
                                 df_success['Affinity_kcal_mol'],
                                 c=df_success['Affinity_kcal_mol'],
                                 cmap='RdYlGn_r', alpha=0.6)
    axes[1, 1].set_xlabel('Üretilen Poz Sayısı')
    axes[1, 1].set_ylabel('Affinity (kcal/mol)')
    axes[1, 1].set_title('Affinity vs Poz Sayısı')
    plt.colorbar(scatter, ax=axes[1, 1], label='Affinity')
    
    plt.tight_layout()
    
    # Figürü kaydet
    fig_path = PROJECT_ROOT / 'docking_analysis.png'
    plt.savefig(fig_path, dpi=300, bbox_inches='tight')
    logger.info(f"📊 Analiz grafiği kaydedildi: {fig_path}")
    
    plt.show()
    
    # İstatistikleri yazdır
    print("\n📈 Özet İstatistikler:")
    print(df_success['Affinity_kcal_mol'].describe())
    
    print("\n🏆 En İyi 10 Ligand:")
    print(df_success.nsmallest(10, 'Affinity_kcal_mol')[['Ligand_Name', 'Affinity_kcal_mol', 'Quality']])
    
    # Top adayları dışa aktar
    top_candidates = df_success.nsmallest(50, 'Affinity_kcal_mol')
    top_candidates_path = PROJECT_ROOT / 'TOP_50_CANDIDATES.csv'
    top_candidates.to_csv(top_candidates_path, index=False)
    logger.info(f"\n💎 En iyi 50 aday dışa aktarıldı: {top_candidates_path}")
    
else:
    print("⚠️ Sonuç dosyası bulunamadı. Lütfen önce docking pipeline'ı çalıştırın.")

## 💾 Bölüm 11: Dış Analiz için Dışa Aktarma

In [ ]:
# Sonuçları farklı formatlarda dışa aktar

if CSV_LOG_PATH.exists():
    df_export = pd.read_csv(CSV_LOG_PATH)
    df_export_success = df_export[df_export['Status'] == 'Başarılı'].copy()
    
    # 1. Excel dışa aktarımı (formatlı)
    excel_path = PROJECT_ROOT / 'DOCKING_RESULTS.xlsx'
    with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
        # Tüm sonuçlar
        df_export_success.to_excel(writer, sheet_name='Tüm Sonuçlar', index=False)
        
        # Top 100
        df_export_success.nsmallest(100, 'Affinity_kcal_mol').to_excel(
            writer, sheet_name='En İyi 100', index=False
        )
        
        # Mükemmel bağlananlar
        df_export_success[df_export_success['Quality'] == 'Mükemmel'].to_excel(
            writer, sheet_name='Mükemmel', index=False
        )
        
        # Özet istatistikler
        summary = df_export_success['Affinity_kcal_mol'].describe().to_frame('Değer')
        summary.to_excel(writer, sheet_name='İstatistikler')
    
    logger.info(f"📊 Excel dışa aktarıldı: {excel_path}")
    
    # 2. En iyi 50 için PDBQT dosya listesi oluştur
    top_50 = df_export_success.nsmallest(50, 'Affinity_kcal_mol')
    pdbqt_list_path = PROJECT_ROOT / 'TOP_50_PDBQT_FILES.txt'
    with open(pdbqt_list_path, 'w') as f:
        f.write("# İleri analiz için en iyi 50 docked poz\n")
        f.write(f"# Hedef: {TARGET_PDB}\n")
        f.write(f"# Oluşturulma: {datetime.now().strftime('%Y-%m-%d')}\n\n")
        for _, row in top_50.iterrows():
            f.write(f"{row['Output_File']}\t{row['Affinity_kcal_mol']:.2f}\t{row['Ligand_Name']}\n")
    
    logger.info(f"📝 PDBQT listesi: {pdbqt_list_path}")
    
    print("\n✅ Dışa aktarma tamamlandı!")
    print(f"\n📦 Tüm dosyalar burada: {PROJECT_ROOT}")
    print("\nŞimdi şunları yapabilirsiniz:")
    print("  1. Dosyaları Google Drive'dan indirin")
    print("  2. En iyi pozları PyMOL/Chimera'da görselleştirin")
    print("  3. İleri analiz yapın (MD simülasyonları, vb.)")
    
else:
    print("⚠️ Dışa aktarım için sonuç dosyası bulunamadı.")

---

## 📚 Notlar ve Öneriler

### ZINC Veritabanı Kullanımı
- **ZINC15**: https://zinc15.docking.org
- **API Erişimi**: Rate limiting'e dikkat edin (1-2 saniye bekleme)
- **Popüler Subsetler**:
  - `fda`: FDA onaylı ilaçlar (~3,000 molekül)
  - `world`: Dünya çapında onaylı ilaçlar (~10,000)
  - `investigational`: Araştırma aşamasında
  - `lead-like`: Lead-like özellikler
  - `fragment`: Fragment kütüphanesi

### Devam Etme Özelliği
Google Colab bağlantısı kesilirse:
1. Bölüm 1'i yeniden çalıştırın (Drive + Kurulum)
2. Bölüm 6'yı yeniden çalıştırın (Hedef hazırlama)
3. Bölüm 9'a atlayın - otomatik olarak son noktadan devam edecek

### Performans İpuçları
- **Exhaustiveness 16**: Dengeli kalite/hız (~30-45s/ligand)
- **Exhaustiveness 8**: Hızlı tarama (~15-20s/ligand)
- **Exhaustiveness 32**: Yüksek doğruluk (~90-120s/ligand)

### Validasyon
Yayın öncesi mutlaka:
1. En iyi 10-20 pozu görsel olarak inceleyin
2. Bağlanma bölgesi etkileşimlerini doğrulayın
3. Exhaustiveness=32 ile yeniden docking düşünün
4. En iyi adaylar üzerinde MD simülasyonları yapın

### Sorun Giderme
- **Bellek yetersizliği**: BATCH_SIZE'ı 50'ye düşürün
- **Timeout hataları**: Timeout'u artırın veya exhaustiveness'i azaltın
- **Çok fazla başarısız docking**: Ligand hazırlama ve grid box boyutunu kontrol edin
- **ZINC indirme hataları**: Rate limiting ve internet bağlantısını kontrol edin

### Ligand Hazırlama Notları
- 3D koordinatlar otomatik olarak üretilir (RDKit ile)
- pH 7.4'te protonation durumu ayarlanır
- Gasteiger kısmi yükleri eklenir
- Torsion tree otomatik hesaplanır

---

**İletişim:** Ersin - Istanbul Health & Technology University

**Atıf:** Bu pipeline'ı kullanırsanız, lütfen AutoDock Vina'yı atıf gösterin:
> Eberhardt, J., Santos-Martins, D., Tillack, A.F., Forli, S. (2021). 
> AutoDock Vina 1.2.0: New Docking Methods, Expanded Force Field, and Python Bindings. 
> Journal of Chemical Information and Modeling.

**ZINC Atfı:**
> Irwin, J.J., Tang, K.G., Young, J., Dandarchuluun, C., Wong, B.R., Khurelbaatar, M., Moroz, Y.S., Mayfield, J., Sayle, R.A. (2020).
> ZINC20 - A Free Ultralarge-Scale Chemical Database for Ligand Discovery.
> Journal of Chemical Information and Modeling.

---